In [0]:


-- 1. 删除旧表
DROP TABLE IF EXISTS adhyivy.default.gold_dim_province;

-- 2. 创建新表
CREATE TABLE adhyivy.default.gold_dim_province 
USING DELTA
--LOCATION 'abfss://bronze@sahyivy.dfs.core.windows.net/gold/dim_province'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',  -- 开启，因为每天全量写入
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.dataSkippingNumIndexedCols' = '10',  -- 提高数据跳过效率
    -- 全量表可以保留更短的日志
    'delta.logRetentionDuration' = 'interval 7 days'
)
AS
select
    province.id,
    province.name,
    province.area_code,
    province.iso_code,
    province.iso_3166_2,
    region_id,
    region_name,
    CURRENT_TIMESTAMP() as load_time
from
(
    select
        id,
        name,
        region_id,
        area_code,
        iso_code,
        iso_3166_2
    from silver_base_province
)province
left join
(
    select
        id,
        region_name
    from silver_base_region
)region
on province.region_id=region.id;